# 1. Persiapan Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

print("Library berhasil diimpor.")

Library berhasil diimpor.


# 2. Upload Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
df = pd.read_csv("dataset_dummy_300.csv")

print("Dataset berhasil dimuat.")

# 3. Melihat Dataset

In [ ]:
df.head()

In [ ]:
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

# 4. Memeriksa Nama Kolom

In [ ]:
print(df.columns.tolist())

# 5. KDD: SELECTION

In [ ]:
feature_columns = [
    'M1', 'M2', 'M3', 'M4', 'M5',
    'M6', 'M7', 'M8', 'M9', 'M10',

    'GB1', 'GB2', 'GB3', 'GB4', 'GB5',
    'GB6', 'GB7', 'GB8', 'GB9', 'GB10',

    'TK1', 'TK2', 'TK3', 'TK4', 'TK5',
    'TK6', 'TK7', 'TK8', 'TK9', 'TK10'
]

X = df[feature_columns]
y = df['Label']

print("Jumlah fitur:", X.shape[1])
print("Jumlah data:", X.shape[0])

# 6. KDD: PREPROCESSING

Cek Missing Value

In [ ]:
print("Jumlah missing value:")
print(X.isnull().sum().sum())

Cek Target

In [ ]:
print("Missing value pada target:", y.isnull().sum())

Cek Duplikat

In [ ]:
print("Jumlah data duplikat:", df.duplicated().sum())

Cek Nilai Jawaban

In [ ]:
for col in feature_columns:
    print(col, sorted(df[col].unique()))

# 7. Melihat Distribusi Label

In [ ]:
print(y.value_counts())

In [ ]:
import seaborn as sns
plt.figure(figsize=(10, 5))

sns.countplot(x=y)

plt.title("Distribusi Profil Kompetensi")
plt.xlabel("Profil Kompetensi")
plt.ylabel("Jumlah Data")

plt.show()

# 8. KDD: TRANSFORMATION

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Mapping label:")
for i, label in enumerate(label_encoder.classes_):
    print(i, "=", label)

In [ ]:
print(y_encoded[:10])

# 9. Membagi Dataset

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Jumlah data training:", len(X_train))
print("Jumlah data testing:", len(X_test))

# 10. Membuat Model Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(X_train, y_train)

print("Decision Tree berhasil dilatih.")

# 11. Menguji Decision Tree

Prediksi

In [ ]:
y_pred_dt = dt_model.predict(X_test)

Hitung Akurasi

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_dt = accuracy_score(y_test, y_pred_dt)

print("Akurasi Decision Tree:", accuracy_dt)

# 12. Evaluasi Lengkap

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_dt,
    target_names=label_encoder.classes_
))

# 13. Membuat Model XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train, y_train)

print("XGBoost berhasil dilatih.")

Prediksi

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

Akurasi

In [ ]:
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print("Akurasi XGBoost:", accuracy_xgb)

# 14. Membandingkan Model

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'Precision': [
        precision_score(y_test, y_pred_dt, average='weighted'),
        precision_score(y_test, y_pred_xgb, average='weighted')
    ],
    'Recall': [
        recall_score(y_test, y_pred_dt, average='weighted'),
        recall_score(y_test, y_pred_xgb, average='weighted')
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_dt, average='weighted'),
        f1_score(y_test, y_pred_xgb, average='weighted')
    ]
})

comparison

# 15. K-Fold Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

Decision Tree

In [ ]:
from sklearn.model_selection import cross_val_score

dt_scores = cross_val_score(
    dt_model,
    X,
    y_encoded,
    cv=kfold,
    scoring='accuracy'
)

print("Decision Tree")
print("Skor setiap fold:", dt_scores)
print("Rata-rata:", dt_scores.mean())

XGBoost

In [ ]:
xgb_scores = cross_val_score(
    xgb_model,
    X,
    y_encoded,
    cv=kfold,
    scoring='accuracy'
)

print("XGBoost")
print("Skor setiap fold:", xgb_scores)
print("Rata-rata:", xgb_scores.mean())

# 16. Confusion Matrix

Decision Tree

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm_dt = confusion_matrix(y_test, y_pred_dt)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm_dt,
    annot=True,
    fmt='d',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix - Decision Tree")

plt.show()

XGBoost

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Membuat confusion matrix untuk XGBoost
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

# Menampilkan confusion matrix
plt.figure(figsize=(8, 6))

sns.heatmap(
    cm_xgb,
    annot=True,
    fmt='d',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix - XGBoost")

plt.show()

# 17. SHAP

In [ ]:
!pip install shap -q

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
feature_names = X_test.columns.tolist()
print(feature_names)

Membuat SHAP Explainer untuk XGBoost

In [ ]:
explainer = shap.TreeExplainer(xgb_model)

In [ ]:
shap_values = explainer.shap_values(X_test)

In [ ]:
explainer = shap.TreeExplainer(xgb_model)

shap_values = explainer.shap_values(X_test)

print("SHAP berhasil dihitung.")

Melihat Bentuk SHAP Values

In [ ]:
print(type(shap_values))

# Assuming shap_values is a 3D numpy array (n_samples, n_features, n_classes)
# If it's a list, the original 'if' block would handle it.
# Since it's an ndarray of shape (60, 30, 6), we handle it here.
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    num_classes = shap_values.shape[2]
    num_samples = shap_values.shape[0]
    num_features = shap_values.shape[1]
    print(f"Jumlah kelas: {num_classes}")
    for i in range(num_classes):
        print(f"Kelas {i}: ({num_samples}, {num_features})")
elif isinstance(shap_values, list):
    print("Jumlah kelas:", len(shap_values))
    for i, values in enumerate(shap_values):
        print(f"Kelas {i}: {values.shape}")
else:
    print("Shape SHAP Values:", shap_values.shape)

Menampilkan SHAP Feature Importance Secara Global

In [ ]:
# Menggabungkan SHAP values dari seluruh kelas
shap_values_all = np.array(shap_values)

# Menghitung rata-rata nilai absolut SHAP
# Perbaiki: Rata-rata harus dihitung di seluruh sampel (axis=0) dan kelas (axis=2) untuk mendapatkan satu nilai per fitur
mean_shap = np.mean(np.abs(shap_values_all), axis=(0, 2))

# Membuat DataFrame feature importance
shap_importance = pd.DataFrame({
    'Fitur': feature_names,
    'Mean Absolute SHAP': mean_shap
})

# Mengurutkan dari yang paling berpengaruh
shap_importance = shap_importance.sort_values(
    by='Mean Absolute SHAP',
    ascending=False
)

print(shap_importance)

Visualisasi Feature Importance SHAP

In [ ]:
plt.figure(figsize=(10, 8))

plt.barh(
    shap_importance['Fitur'].head(15)[::-1],
    shap_importance['Mean Absolute SHAP'].head(15)[::-1]
)

plt.xlabel('Mean Absolute SHAP Value')
plt.ylabel('Fitur')
plt.title('Top 15 Feature Importance berdasarkan SHAP')

plt.show()

SHAP Summary Plot

In [ ]:
# Menampilkan summary plot
# shap.summary_plot can handle the 3D shap_values directly for multi-class models
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=feature_names
)

SHAP untuk Menganalisis Profil Tertentu

In [ ]:
# contoh
target_class = 2

# Correctly slice shap_values to get SHAP values for the target class
# Assuming shap_values is (n_samples, n_features, n_classes)
shap_pc03 = shap_values[:, :, target_class]

shap.summary_plot(
    shap_pc03,
    X_test,
    feature_names=feature_names
)

SHAP untuk Semua Profil Kompetensi

In [ ]:
class_names = [
    'PC01',
    'PC02',
    'PC03',
    'PC04',
    'PC05',
    'PC06'
]

for i, class_name in enumerate(class_names):

    print(f"\n===== SHAP FEATURE IMPORTANCE {class_name} =====")

    # Fix: Correctly slice shap_values to get SHAP values for the target class
    # Assuming shap_values is (n_samples, n_features, n_classes)
    shap_class = shap_values[:, :, i]

    mean_abs_shap_class = np.mean(
        np.abs(shap_class),
        axis=0
    )

    importance_class = pd.DataFrame({
        'Fitur': feature_names,
        'Mean Absolute SHAP': mean_abs_shap_class
    })

    importance_class = importance_class.sort_values(
        by='Mean Absolute SHAP',
        ascending=False
    )

    print(
        importance_class.head(10).to_string(index=False)
    )

Visualisasi SHAP untuk Setiap Profil

In [ ]:
# PC01

target_class = 0

# Correctly slice shap_values to get SHAP values for the target class
# Assuming shap_values is (n_samples, n_features, n_classes)
shap_class = shap_values[:, :, target_class]

shap.summary_plot(
    shap_class,
    X_test,
    feature_names=feature_names,
    max_display=10
)

In [ ]:
# PC02

target_class = 1

# Correctly slice shap_values to get SHAP values for the target class
# Assuming shap_values is (n_samples, n_features, n_classes)
shap_class = shap_values[:, :, target_class]

shap.summary_plot(
    shap_class,
    X_test,
    feature_names=feature_names,
    max_display=10
)

In [ ]:
# PC03

target_class = 2

# Correctly slice shap_values to get SHAP values for the target class
# Assuming shap_values is (n_samples, n_features, n_classes)
shap_class = shap_values[:, :, target_class]

shap.summary_plot(
    shap_class,
    X_test,
    feature_names=feature_names,
    max_display=10
)

In [ ]:
# PC04

target_class = 3

# Correctly slice shap_values to get SHAP values for the target class
# Assuming shap_values is (n_samples, n_features, n_classes)
shap_class = shap_values[:, :, target_class]

shap.summary_plot(
    shap_class,
    X_test,
    feature_names=feature_names,
    max_display=10
)

In [ ]:
# PC05

target_class = 4

# Correctly slice shap_values to get SHAP values for the target class
# Assuming shap_values is (n_samples, n_features, n_classes)
shap_class = shap_values[:, :, target_class]

shap.summary_plot(
    shap_class,
    X_test,
    feature_names=feature_names,
    max_display=10
)

In [ ]:
# PC06

target_class = 5

# Correctly slice shap_values to get SHAP values for the target class
# Assuming shap_values is (n_samples, n_features, n_classes)
shap_class = shap_values[:, :, target_class]

shap.summary_plot(
    shap_class,
    X_test,
    feature_names=feature_names,
    max_display=10
)

SHAP Individual — Menjelaskan Satu Mahasiswa

In [ ]:
sample_index = 0

sample = X_test.iloc[[sample_index]]

print(sample)

In [ ]:
prediction = xgb_model.predict(sample)

print("Predicted Class:", prediction[0])

In [ ]:
predicted_class = int(prediction[0])

# Fix: Correctly slice shap_values for a specific sample and predicted class
# shap_values is (n_samples, n_features, n_classes)
sample_shap = shap_values[sample_index, :, predicted_class]

sample_shap_df = pd.DataFrame({
    'Fitur': feature_names,
    'SHAP Value': sample_shap,
    'Nilai Jawaban': sample.iloc[0].values
})

sample_shap_df['Abs SHAP'] = np.abs(
    sample_shap_df['SHAP Value']
)

sample_shap_df = sample_shap_df.sort_values(
    by='Abs SHAP',
    ascending=False
)

print(
    sample_shap_df.head(10).to_string(index=False)
)

# Menyimpan Model

In [ ]:
import joblib

joblib.dump(
    xgb_model,
    "model_profil_kompetensi.pkl"
)

joblib.dump(
    label_encoder,
    "label_encoder.pkl"
)

print("Model berhasil disimpan.")

In [ ]:
model = joblib.load("model_profil_kompetensi.pkl")

## Simpan model di google colab

In [ ]:
import joblib

# 1. Simpan model ML yang sudah dilatih (misal nama variabel model kamu: 'model')
joblib.dump(model, 'model_cpl.pkl')

# 2. Jika kamu memakai LabelEncoder atau Scaler, simpan juga!
# joblib.dump(scaler, 'scaler.pkl')

print("Model berhasil disimpan!")

In [ ]:
from google.colab import files

# Download file model
files.download('model_cpl.pkl')